In [ ]:
import pandas as pd
import json

# -----------------------------
# 1. Cargar CSV
# -----------------------------
df = pd.read_csv("src\data\df_final.csv")


# -----------------------------
# 2. Configuración
# -----------------------------
variable_objetivo = "colseg"
categorias_colseg = ["Peor", "Igual", "Mejor"]

variables = [col for col in df.columns if col not in [variable_objetivo, "idnum", ]]

df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64").astype(str)

# Orden específico para ideología
orden_ideologia = ["Izquierda", "Centro izquierda", "Centro", "Centro derecha", "Derecha"]

if "ideologia" in df.columns:
    df["ideologia"] = pd.Categorical(df["ideologia"], categories=orden_ideologia, ordered=True)

# Ajuste grupos edad

df["q2"] = pd.to_numeric(df["q2"], errors="coerce")
df["q2"] = (df["q2"] // 10).astype("Int64")

df["q2"] = (
    "Entre " +
    (df["q2"] * 10).astype("Int64").astype(str) +
    " y " +
    (df["q2"] * 10 + 9).astype("Int64").astype(str)
)



# -----------------------------
# 3. Crear JSONs
# -----------------------------
json_absoluto = {}
json_relativo = {}

for variable in variables:

    json_absoluto[variable] = {}
    json_relativo[variable] = {}

    for anio in df["year"].unique():

        df_anio = df[df["year"] == anio]

        temp = df_anio[[variable, variable_objetivo]].copy()
        temp = temp.dropna(subset=[variable, variable_objetivo])

        temp[variable] = temp[variable].astype(str)
        temp[variable_objetivo] = temp[variable_objetivo].astype(str)

        # -------------------------
        # ABSOLUTO
        # -------------------------
        tabla_abs = pd.crosstab(temp[variable], temp[variable_objetivo])
        tabla_abs = tabla_abs.reindex(columns=categorias_colseg, fill_value=0)

        # Forzar orden en ideología
        if variable == "ideologia":
            tabla_abs = tabla_abs.reindex(orden_ideologia)

        json_absoluto[variable][anio] = {}

        for categoria in tabla_abs.index:
            json_absoluto[variable][anio][categoria] = {
                "Peor": int(tabla_abs.loc[categoria, "Peor"]),
                "Igual": int(tabla_abs.loc[categoria, "Igual"]),
                "Mejor": int(tabla_abs.loc[categoria, "Mejor"])
            }

        # -------------------------
        # RELATIVO
        # -------------------------
        tabla_rel = tabla_abs.div(tabla_abs.sum(axis=1), axis=0).fillna(0)

        json_relativo[variable][anio] = {}

        for categoria in tabla_rel.index:
            json_relativo[variable][anio][categoria] = {
                "Peor": float(tabla_rel.loc[categoria, "Peor"]),
                "Igual": float(tabla_rel.loc[categoria, "Igual"]),
                "Mejor": float(tabla_rel.loc[categoria, "Mejor"])
            }

# -----------------------------
# 4. Guardar
# -----------------------------
with open("src\data\heatmap_absoluto.json", "w", encoding="utf-8") as f:
    json.dump(json_absoluto, f, ensure_ascii=False, indent=2)

with open("src\data\heatmap_relativo.json", "w", encoding="utf-8") as f:
    json.dump(json_relativo, f, ensure_ascii=False, indent=2)

print("Listo.")

<>:7: SyntaxWarning: invalid escape sequence '\O'
<>:97: SyntaxWarning: invalid escape sequence '\O'
<>:100: SyntaxWarning: invalid escape sequence '\O'
<>:7: SyntaxWarning: invalid escape sequence '\O'
<>:97: SyntaxWarning: invalid escape sequence '\O'
<>:100: SyntaxWarning: invalid escape sequence '\O'
C:\Users\diego\AppData\Local\Temp\ipykernel_1168\1914107072.py:7: SyntaxWarning: invalid escape sequence '\O'
  df = pd.read_csv("Página tesis\Observatorio\src\data\df_final.csv")
C:\Users\diego\AppData\Local\Temp\ipykernel_1168\1914107072.py:97: SyntaxWarning: invalid escape sequence '\O'
  with open("Página tesis\Observatorio\src\data\heatmap_absoluto.json", "w", encoding="utf-8") as f:
C:\Users\diego\AppData\Local\Temp\ipykernel_1168\1914107072.py:100: SyntaxWarning: invalid escape sequence '\O'
  with open("Página tesis\Observatorio\src\data\heatmap_relativo.json", "w", encoding="utf-8") as f:


Listo.
